# Gu & Kurov (2020) — DoubleML PLR Robustness
**Ashley Thompson — Capstone**

DoubleML Partially Linear Regression (Robinson 1988) robustness analysis for the Gu & Kurov (2020) replication. XGBoost nuisance models partial out nonlinear confounding from the Gu & Kurov control set so treatment coefficients have a causal interpretation under conditional unconfoundedness.

**Treatment channels:**
- `twitter_sent` — Bloomberg composite Twitter sentiment
- `news_sent` — Bloomberg composite news sentiment

**Outcome:** `return_oo` — open-to-open holding-period return, %  
`return_oo = (px_open_{t+1} - px_open_t) / px_open_t × 100`

**Confounders (Gu & Kurov specification):**  
Rogers-Satchell (1991) realized volatility, abnormal trading volume, log(market cap), bid-ask spread (if available), total equity, D/E ratio, RSI-30, MA-50, tweet count.

**Structure:**
1. Main ATE — current treatment → return_oo
2. Part 1 — lagged treatment → return_oo *(reverse causality check)*
3. No-reversal test — all sentiment lags simultaneously *(Gu & Kurov 2020 Table 3 analog)*
4. Part 2 — current treatment → return_oo_lead{n} *(impact persistence)*
5. Export LaTeX tables

> **Runtime → Change runtime type → T4 GPU** before running.

## 1. Install dependencies

In [ ]:
!pip install -q doubleml xgboost scikit-learn

## 2. Load data
Run Option A or Option B, then the GitHub setup cell.

In [ ]:
# Option A — Google Drive (data only)
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Colab Notebooks/Capstone/panel_long.csv'

In [ ]:
# Option B — Manual upload (data only)
from google.colab import files
uploaded = files.upload()
DATA_PATH = 'panel_long.csv'

In [ ]:
# GitHub output setup — run after whichever data option above
from google.colab import userdata
import os, subprocess

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_DIR     = '/content/Capstone'
OUTPUT_PATH  = REPO_DIR + '/output/'

if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(
        ['git', 'clone',
         f'https://{GITHUB_TOKEN}@github.com/ATHOMP-03/Capstone.git',
         REPO_DIR],
        check=True,
    )
os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f'Output path ready: {OUTPUT_PATH}')

## 3. Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import doubleml as dml
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')
np.random.seed(42)
os.makedirs(OUTPUT_PATH, exist_ok=True)

long = pd.read_csv(DATA_PATH, parse_dates=['date'])
long = long.sort_values(['ticker', 'date']).reset_index(drop=True)
print(f'Loaded {len(long):,} rows x {long.shape[1]} cols')

HAS_SPREAD = 'bid_ask_spread' in long.columns
print(f'HAS_SPREAD = {HAS_SPREAD}  (bid-ask spread in confounder set: {HAS_SPREAD})')
long.head()

In [ ]:
# Variable construction — mirrors gu_kurov_replication.py

# Open-to-open return
long['px_open_next'] = long.groupby('ticker')['px_open'].shift(-1)
long['return_oo'] = (long['px_open_next'] - long['px_open']) / long['px_open'] * 100

# Rogers-Satchell (1991) realized volatility
log_H = np.log(long['px_high'].clip(lower=1e-8))
log_L = np.log(long['px_low'].clip(lower=1e-8))
log_O = np.log(long['px_open'].clip(lower=1e-8))
log_C = np.log(long['px_close'].clip(lower=1e-8))
long['vol_rs'] = ((log_H - log_C) * (log_H - log_O) + (log_L - log_C) * (log_L - log_O)).clip(lower=0) * 100

# Abnormal trading volume
mean_vol = long.groupby('ticker')['volume'].transform('mean')
long['abnorm_vol'] = ((long['volume'] - mean_vol) / mean_vol) * 100

# Log market cap
long['log_mkt_cap'] = np.log(long['mkt_cap'].clip(lower=1e-8))

print(f"return_oo: mean={long['return_oo'].mean():.4f}  std={long['return_oo'].std():.4f}")
print(f"vol_rs:    mean={long['vol_rs'].mean():.4f}")
print(f"abnorm_vol: mean={long['abnorm_vol'].mean():.4f}")

In [ ]:
def make_xgb():
    return XGBRegressor(
        n_estimators     = 500,
        learning_rate    = 0.05,
        max_depth        = 6,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        device           = 'cuda',
        tree_method      = 'hist',
        random_state     = 42,
        n_jobs           = -1,
    )

LAGS = [1, 2, 3, 5, 7]

# Gu & Kurov confounder set — excludes both treatment channels
GK_CONFOUNDERS = [
    'vol_rs', 'abnorm_vol', 'log_mkt_cap',
    'total_equity', 'debt_to_equity',
    'twitter_count', 'rsi_30', 'ma_50',
]
if HAS_SPREAD:
    GK_CONFOUNDERS.append('bid_ask_spread')

TREATMENTS = [
    ('twitter_sent', 'Twitter sentiment'),
    ('news_sent',    'News sentiment'),
]

def run_plr(df, y_col, d_col, x_cols):
    data  = dml.DoubleMLData(
        df[[y_col, d_col] + x_cols].dropna().reset_index(drop=True),
        y_col=y_col, d_cols=d_col, x_cols=x_cols,
    )
    model = dml.DoubleMLPLR(data, ml_l=make_xgb(), ml_m=make_xgb(), n_folds=5, n_rep=20)
    model.fit()
    model.bootstrap(method='normal', n_rep_boot=1000)
    coef     = model.summary['coef'].iloc[0]
    pval     = model.summary['P>|t|'].iloc[0]
    ci_lower = model.confint(level=0.95)['2.5 %'].iloc[0]
    ci_upper = model.confint(level=0.95)['97.5 %'].iloc[0]
    return coef, pval, ci_lower, ci_upper

def run_plr_multi(df, y_col, d_cols, x_cols):
    keep  = [y_col] + d_cols + x_cols
    data  = dml.DoubleMLData(
        df[keep].dropna().reset_index(drop=True),
        y_col=y_col, d_cols=d_cols, x_cols=x_cols,
    )
    model = dml.DoubleMLPLR(data, ml_l=make_xgb(), ml_m=make_xgb(), n_folds=5, n_rep=20)
    model.fit()
    model.bootstrap(method='normal', n_rep_boot=1000)
    return model

print('Setup complete.')

## 4. Main ATE — current treatment → return_oo
Primary causal estimate. Each sentiment channel estimated separately with the full Gu & Kurov confounder set partialled out by XGBoost.

In [ ]:
main_rows = []

for col, desc in TREATMENTS:
    coef, pval, ci_lower, ci_upper = run_plr(long, 'return_oo', col, GK_CONFOUNDERS)
    sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
    print(f'{desc:<30} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')
    main_rows.append({'label': desc, 'coef': coef, 'pval': pval, 'ci_lower': ci_lower, 'ci_upper': ci_upper})

main_df = pd.DataFrame(main_rows)
print('\nMAIN ATE TABLE')
print(main_df.to_string(index=False))

## 5. Part 1 — Lagged treatment → return_oo
*(Reverse causality check — univariate)*

Tests whether past sentiment predicts current open-to-open returns. Under the Gu & Kurov hypothesis, only lag 1 should be significant (the market incorporates the signal in one trading day).

**Pass:** lags 3–7 insignificant.

In [ ]:
# Build lagged treatment columns
for col, _ in TREATMENTS:
    for lag_n in LAGS:
        long[f'{col}_lag{lag_n}'] = long.groupby('ticker')[col].shift(lag_n)

part1_rows = []

for lag_n in LAGS:
    print(f'\n--- Lag {lag_n} ---')
    for col, desc in TREATMENTS:
        lag_col = f'{col}_lag{lag_n}'
        coef, pval, ci_lower, ci_upper = run_plr(long, 'return_oo', lag_col, GK_CONFOUNDERS)
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
        print(f'  lag{lag_n} | {desc:<25} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')
        part1_rows.append({'label': desc, 'lag': lag_n, 'coef': coef, 'pval': pval, 'ci_lower': ci_lower, 'ci_upper': ci_upper})

part1_df = pd.DataFrame(part1_rows)
print('\nLAG DECAY TABLE (univariate)')
print(part1_df.sort_values(['lag', 'label']).to_string(index=False))

## 6. No-reversal test — all sentiment lags simultaneously
*(Gu & Kurov 2020 Table 3 analog under DoubleML)*

All lags entered jointly; each coefficient is conditioned on the others. This is the direct analog of the Gu & Kurov no-reversal specification reframed under double machine learning.

**Pass:** lag 1 significant, lags 2–7 insignificant → information absorbed within one trading day  
**Fail:** lags 3–7 persist → sentiment driven by price momentum (reverse causality)

In [ ]:
for col, desc in TREATMENTS:
    lag_cols = [f'{col}_lag{n}' for n in LAGS]

    nr_model   = run_plr_multi(long, 'return_oo', lag_cols, GK_CONFOUNDERS)
    nr_summary = nr_model.summary
    nr_confint = nr_model.confint(level=0.95)

    print(f'\n--- {desc} ---')
    verdicts = {}
    for lag_col in lag_cols:
        coef     = nr_summary.loc[lag_col, 'coef']
        pval     = nr_summary.loc[lag_col, 'P>|t|']
        ci_lower = nr_confint.loc[lag_col, '2.5 %']
        ci_upper = nr_confint.loc[lag_col, '97.5 %']
        verdicts[lag_col] = pval < 0.05
        sig = '*' if pval < 0.05 else ''
        print(f'  {lag_col:<25} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')

    lag1_col  = f'{col}_lag1'
    later_sig = [c for c in lag_cols if c != lag1_col and verdicts.get(c, False)]
    print(f'\n  VERDICT ({desc}):')
    if verdicts.get(lag1_col) and not later_sig:
        print('  PASS — lag 1 significant, lags 2-7 insignificant.')
        print('         Consistent with Gu & Kurov (2020): information absorbed within one trading day.')
    elif not verdicts.get(lag1_col):
        print('  INCONCLUSIVE — lag 1 not significant.')
    else:
        print(f'  FAIL — significant persistence at: {", ".join(later_sig)}.')
        print('         Suggests price-driven sentiment (return → tweets). Interpret ATE with caution.')

## 7. Part 2 — Current treatment → return_oo_lead{n}
*(Impact persistence — research question)*

How long does today's sentiment shift affect future open-to-open returns?

Under semi-strong efficiency, Twitter signals should be absorbed within 1–2 trading days. Persistence beyond day 3 implies drift — a meaningful departure from the Gu & Kurov framework.

In [ ]:
# Create lead return variables
for lead_n in LAGS:
    long[f'return_oo_lead{lead_n}'] = long.groupby('ticker')['return_oo'].shift(-lead_n)

part2_rows = []

for lead_n in LAGS:
    outcome = f'return_oo_lead{lead_n}'
    print(f'\n--- Lead {lead_n} ---')
    for col, desc in TREATMENTS:
        coef, pval, ci_lower, ci_upper = run_plr(long, outcome, col, GK_CONFOUNDERS)
        sig = '***' if pval < 0.01 else '**' if pval < 0.05 else '*' if pval < 0.10 else ''
        print(f'  lead{lead_n} | {desc:<25} coef={coef:>9.6f}  p={pval:.4f}  CI=[{ci_lower:.6f}, {ci_upper:.6f}]  {sig}')
        part2_rows.append({'label': desc, 'lead': lead_n, 'coef': coef, 'pval': pval, 'ci_lower': ci_lower, 'ci_upper': ci_upper})

part2_df = pd.DataFrame(part2_rows)
print('\nIMPACT PERSISTENCE TABLE')
print(part2_df.sort_values(['lead', 'label']).to_string(index=False))
print('\n* If twitter_sent insignificant by lead 2-3 but news_sent persists,')
print('  tweet cycles are shorter than news cycles — consistent with Gu & Kurov (2020).')

## 8. Export — LaTeX tables

In [ ]:
def to_latex(rows, caption, label, period_col=''):
    def stars(p):
        if p < 0.01: return '^{***}'
        if p < 0.05: return '^{**}'
        if p < 0.10: return '^{*}'
        return ''

    period_hdr = f' & {period_col.title()}' if period_col else ''
    tab_spec   = r'{llcccc}' if period_col else r'{lcccc}'
    spread_note = ' Bid-ask spread included.' if HAS_SPREAD else ' Bid-ask spread omitted (unavailable).'
    lines = [
        r'\begin{table}[htbp]', r'\centering',
        f'\\caption{{{caption}}}', f'\\label{{{label}}}',
        f'\\begin{{tabular}}{tab_spec}',
        r'\hline\hline',
        f'Treatment{period_hdr} & Coefficient & p-value & 95\\% CI Lower & 95\\% CI Upper \\\\',
        r'\hline',
    ]
    for r in rows:
        period_cell = f' & {int(r[period_col])}' if period_col else ''
        lbl = str(r['label']).replace('_', r'\_')
        lines.append(
            f"{lbl}{period_cell} & ${r['coef']:.6f}{stars(r['pval'])}$ "
            f"& {r['pval']:.4f} & {r['ci_lower']:.6f} & {r['ci_upper']:.6f} \\\\"
        )
    lines += [
        r'\hline',
        r'\multicolumn{5}{l}{\footnotesize DoubleML PLR, XGBoost (GPU), 5-fold, 20 reps, 1000 bootstrap draws.} \\',
        f'\\multicolumn{{5}}{{l}}{{\\footnotesize Confounders: RS vol, abnorm vol, log mkt cap, D/E, RSI, MA50, tweet count.{spread_note}}} \\\\',
        r'\multicolumn{5}{l}{\footnotesize $^{***}p<0.01$, $^{**}p<0.05$, $^{*}p<0.10$.} \\',
        r'\hline\hline', r'\end{tabular}', r'\end{table}',
    ]
    return '\n'.join(lines)


# Main ATE
with open(OUTPUT_PATH + 'gk_main_ate.tex', 'w') as f:
    f.write(to_latex(main_rows, 'Gu \\& Kurov (2020) --- DoubleML PLR Main ATE Estimates', 'tab:gk_main_ate'))
print('Saved gk_main_ate.tex')

# Lag decay (Part 1)
with open(OUTPUT_PATH + 'gk_lag_decay.tex', 'w') as f:
    f.write(to_latex(
        sorted(part1_rows, key=lambda r: (r['lag'], r['label'])),
        'Gu \\& Kurov (2020) --- Lagged Sentiment and Current Open-to-Open Return (Reverse Causality Check)',
        'tab:gk_lag_decay', period_col='lag',
    ))
print('Saved gk_lag_decay.tex')

# Impact persistence (Part 2)
with open(OUTPUT_PATH + 'gk_impact_persistence.tex', 'w') as f:
    f.write(to_latex(
        sorted(part2_rows, key=lambda r: (r['lead'], r['label'])),
        'Gu \\& Kurov (2020) --- Current Sentiment and Future Open-to-Open Return (Impact Persistence)',
        'tab:gk_impact_persistence', period_col='lead',
    ))
print('Saved gk_impact_persistence.tex')

In [ ]:
# Push outputs to GitHub
import os, subprocess

os.chdir(REPO_DIR)
subprocess.run(['git', 'config', 'user.email', 'ATHOMP-03@users.noreply.github.com'])
subprocess.run(['git', 'config', 'user.name',  'ATHOMP-03'])
subprocess.run(['git', 'add', 'output/'])

staged   = subprocess.run(['git', 'diff', '--cached', '--quiet'])
unpushed = subprocess.run(
    ['git', 'log', 'origin/main..HEAD', '--oneline'],
    capture_output=True, text=True
)

if staged.returncode != 0:
    subprocess.run(['git', 'commit', '-m', 'Add Gu & Kurov DoubleML robustness results from Colab'], check=True)

if staged.returncode != 0 or unpushed.stdout.strip():
    subprocess.run(['git', 'push'], check=True)
    print('Pushed to GitHub.')
else:
    print('Nothing new to commit or push.')